<a href="https://colab.research.google.com/github/hajonghyun/installPytorch_study/blob/main/7_1_mininet_test_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
import torch
from torch import nn

In [83]:
x = torch.randn(100,3)
layer = nn.Linear(3,5)

# print하기 전에 결과 예측해보기
print(layer(x).shape)
print(layer.weight.shape)
print(layer.bias.shape)


torch.Size([100, 5])
torch.Size([5, 3])
torch.Size([5])


# 딥러닝 핵심: ReLU (Rectified Linear Unit)

**ReLU**는 딥러닝 역사상 가장 중요한 발견 중 하나로, 딥러닝 모델을 깊게(Deep) 쌓을 수 있게 만든 핵심 활성화 함수입니다.

---

## 1. 💡 직관적 이해: "깐깐한 문지기"

ReLU는 들어오는 신호를 선별하는 **전기 스위치(Switch)** 또는 **문지기**와 같습니다.

* **양수 ($+$)**: "중요한 신호(Feature)다!" $\rightarrow$ **그대로 통과 (Pass)**
* **음수 ($-$)**: "쓸모없는 노이즈다." $\rightarrow$ **0으로 차단 (Block)**

> **왜 비선형(Non-linear)인가?**
> 세상의 데이터(이미지, 소리)는 직선 하나로 나눌 수 없습니다. 0에서 꺾이는(Rectified) 구조가 있어야 신경망이 복잡한 경계선을 그릴 수 있습니다.

---

## 2. 📐 수식과 그래프

$$f(x) = \max(0, x)$$

* $x > 0$: 기울기(Gradient) = **1**
* $x < 0$: 기울기(Gradient) = **0**

---

## 3. ❓ 핵심 Q&A (헷갈리기 쉬운 포인트)

이번 학습 과정에서 다루었던 중요한 오개념들을 정리합니다.

### Q1. "음수는 왜 불필요한 정보로 취급하나요?"
수학적으로 음수가 나쁜 것은 아니지만, **생물학적/구조적 효율성** 때문입니다.
1.  **생물학적 모방:** 뇌세포(뉴런)는 자극이 없으면 가만히 있지, 음수 신호를 쏘지 않습니다. (Firing or Not)
2.  **특징 추출(Feature Detection):** CNN은 "특징이 있냐/없냐"를 따집니다. 특징의 반대 패턴(음수)은 "특징 없음(0)"으로 처리하는 것이 모델을 단순화시킵니다.
3.  **희소성(Sparsity):** 적당히 많은 뉴런이 0이 되어야(꺼져야), 진짜 중요한 뉴런의 신호가 돋보입니다.

### Q2. "Leaky ReLU는 음수를 양수로 만드는 건가요?"
**아닙니다!** 부호는 그대로 두고 크기만 줄이는 것입니다.
* **오해:** $-100 \times -0.01 = +1$ (양수로 변환? ❌)
* **진실:** $-100 \times +0.01 = -1$ (음수 유지, 크기 축소 ⭕)
    * "아니라는 건 알겠는데(음수), 너무 강하게 부정하지는 마(값 축소)."라는 의미입니다.

---

## 4. 🏆 이론적 핵심: "Why ReLU?"

왜 Sigmoid를 버리고 ReLU를 쓸까요? 바로 **Vanishing Gradient (기울기 소실)** 문제 해결 때문입니다.

| 비교 | Sigmoid | ReLU |
| :--- | :--- | :--- |
| **최대 기울기** | **0.25** (너무 작음) | **1** (양수 구간) |
| **문제점** | 층이 깊어지면 $0.25 \times 0.25 \dots$ 반복하여 기울기가 0이 됨 (학습 마비). | $1 \times 1 \dots = 1$. 층이 아무리 깊어도 **기울기가 그대로 전달됨.** |
| **결과** | 얕은 모델만 가능 | **Deep Neural Network 가능** |

---

## 5. 💻 PyTorch 구현 (Code)

실무에서 사용하는 두 가지 방식입니다.

### 1) `nn.ReLU` (Class 방식)
* 주로 `nn.Sequential` 모델을 쌓을 때 레고 블록처럼 사용합니다.
* `inplace=True`: 메모리 절약을 위해 원본 데이터를 덮어씁니다.

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),  # 레이어로서 존재
    nn.MaxPool2d(2)
)
```

### 2) `F.relu` (Functional 방식)
* `forward` 함수 내에서 수식을 직접 짤 때 사용합니다.

```python
import torch.nn.functional as F

class MyModel(nn.Module):
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)  # 함수처럼 호출
        return x
```

---

## 6. ⚠️ 심화: Dying ReLU & Leaky ReLU

* **Dying ReLU:** 학습 중 뉴런에 계속 음수만 들어오면, 기울기가 0이 되어 영원히 깨어나지 못하는 "죽은 뉴런"이 발생할 수 있습니다.
* **Leaky ReLU:** 이를 방지하기 위해 음수 구간에 아주 미세한 기울기($0.01x$)를 주어 "살려는 드리는" 변형 함수입니다.

In [84]:
x=torch.randn(2,5)
layer=nn.ReLU()
print(x)
print(layer(x))

tensor([[-0.3430,  1.9806, -0.3496, -1.4859,  1.5650],
        [ 1.0775, -0.6704,  1.0000,  1.3798, -1.1741]])
tensor([[0.0000, 1.9806, 0.0000, 0.0000, 1.5650],
        [1.0775, 0.0000, 1.0000, 1.3798, 0.0000]])


# 📚 Batch Normalization (BN) 완전 정복: A to Z

> **핵심 요약:** > "앞사람(이전 Layer)이 데이터를 아무렇게나 던져도, 내가 받아서 **깔끔하게 정렬(Normalize)**한 뒤 **가장 학습하기 좋은 위치로 재배치(Scale & Shift)**해서 뒷사람에게 넘겨준다."

---

## 1. 🏜️ 직관: "모래 뿌리기" (feat. 혁펜하임)

### 💀 문제 상황: Internal Covariate Shift
딥러닝 학습은 이어달리기와 같습니다. 앞단 레이어(Layer)의 가중치($W$)가 계속 변하다 보니, 뒷단으로 넘어오는 데이터(Feature)의 분포가 계속 제멋대로 바뀝니다.
* **비유:** 들것에 실린 모래가 어떨 때는 **왼쪽 구석(음수)**에 쏠리고, 어떨 때는 **사방팔방 퍼져(분산 큼)** 들어옵니다.
* **결과:** 뒷단 레이어는 "아니, 이번엔 또 어디로 튈지 모르겠네?" 하며 당황해서 학습 속도가 느려집니다.

### 🛡️ 해결책: 2단계 재배치
BN은 데이터를 받아서 두 단계로 처리합니다.

1.  **강제 정렬 (Normalization):** 일단 모래를 **중앙(0)**으로 모으고, **적당한 폭(1)**으로 다집니다.
2.  **재배치 (Scale & Shift):** AI에게 권한을 줍니다. "네가 학습하기 제일 좋은 위치와 넓이로 **다시 뿌려봐!**"

---

## 2. 📐 핵심 원리와 수식

BN의 마법은 단순히 0으로 만드는 것(Step 1)이 아니라, **다시 흩트리는 것(Step 2)**에 있습니다.

### Step 1. Normalization (정규화)
들어온 배치 데이터($x$)의 평균과 분산을 구해 표준화합니다.
$$ \hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} $$
* 결과: 평균 0, 분산 1

### Step 2. Scale & Shift (재배치) 🔥 핵심
정규화된 데이터($\hat{x}$)에 학습 가능한 파라미터 $\gamma, \beta$를 적용합니다.
$$ y = \gamma \hat{x} + \beta $$

* **$\gamma$ (Gamma):** Scale (폭 조절) $\rightarrow$ "얼마나 세게 뿌릴까?" (분산 조절)
* **$\beta$ (Beta):** Shift (이동) $\rightarrow$ "어디 좌표에 뿌릴까?" (평균 조절)

---

## 3. 🧠 개념 확인 퀴즈 (Quizzes)

학습한 내용을 점검해 봅시다.

### Q1. 선형성의 함정
> **문제:** 왜 데이터를 정규화(Step 1)만 하고 끝내지 않고, 굳이 $\gamma, \beta$를 써서 다시 값을 망가뜨릴까요?
>
> **정답 및 해설:**
> 데이터를 무조건 0 근처(평균 0, 분산 1)에만 모아두면, **Sigmoid나 ReLU 같은 활성화 함수의 '선형(Linear) 구간'에만 데이터가 갇히게 됩니다.** (Sigmoid의 0 근처는 직선에 가깝습니다.)
> 딥러닝의 핵심은 비선형(구불구불함)을 통해 복잡한 문제를 푸는 것인데, 데이터가 선형 구간에만 있으면 층을 아무리 깊게 쌓아도 단순한 선형 모델이 되어버립니다. 그래서 $\gamma, \beta$를 통해 **"필요하다면 비선형 구간으로 데이터를 밀어버릴 수 있는(Shift)" 융통성**을 주는 것입니다.

### Q2. 파라미터 구분하기
> **문제:** 다음 중 AI가 역전파(Backprop)를 통해 **학습하는 파라미터**는 무엇인가요?
> 1. 배치의 평균($\mu$)과 분산($\sigma^2$)
> 2. 스케일($\gamma$)과 시프트($\beta$)
>
> **정답 및 해설:** **2번 ($\gamma, \beta$)**
> * $\mu, \sigma$: 그냥 들어온 데이터를 보고 계산기 두드려 구한 **통계값**입니다.
> * $\gamma, \beta$: "이 위치가 좋겠어!"라고 AI가 시행착오를 겪며 찾아내는 **학습 변수(Weight/Bias)**입니다.

### Q3. Train vs Test 시나리오
> **문제:** 테스트(Inference) 단계에서 데이터가 1개만 들어왔습니다. 이때 학습 때처럼 그 데이터 1개의 평균과 분산을 구해서 정규화하면 어떻게 될까요?
>
> **정답 및 해설:** **대참사(망함)**가 일어납니다.
> 데이터 1개의 평균은 자기 자신이고 분산은 0입니다. 분모가 0이 되어 에러가 나거나, 값이 0으로 고정되어 모델이 아무것도 예측하지 못합니다. 따라서 테스트 때는 **학습 중에 미리 적어둔 '전체 이동 평균(Running Mean/Var)'**을 꺼내 써야 합니다.

---

## 4. ⚔️ 면접 필살기: 학습 vs 추론

| 구분 | 학습 (Model.train()) | 추론/평가 (Model.eval()) |
| :--- | :--- | :--- |
| **평균/분산 기준** | **현재 들어온 배치(Batch)**의 통계값 사용 | 학습 중 누적된 **이동 평균(Running Stats)** 사용 |
| **특이사항** | 동시에 `Running Mean/Var`를 몰래 업데이트함 (컨닝 페이퍼 작성) | 저장된 `Running Mean/Var`를 꺼내서 정규화함 (컨닝 페이퍼 사용) |
| **주의점** | 배치 사이즈가 너무 작으면(예: 2, 4) 통계가 불안정함 | `state_dict` 저장 시 Running Stats도 반드시 저장해야 함 |

---

## 5. 💻 PyTorch Code 실무 적용

### 기본 코드 분석
```python
import torch.nn as nn

# 채널이 3개인 BN 레이어
bn = nn.BatchNorm1d(3)

# 1. 학습 가능한 파라미터 (Gradient 계산 O)
print(bn.weight) # Gamma (초기값 1) -> Scale
print(bn.bias)   # Beta  (초기값 0) -> Shift

# 2. 학습되지 않는 버퍼 (Gradient 계산 X) -> 테스트 때 사용
print(bn.running_mean) # 이동 평균 (초기값 0)
print(bn.running_var)  # 이동 분산 (초기값 1)
```

### 꿀팁: ConvBlock 구현 시
`Conv2d` 바로 뒤에 `BatchNorm`을 쓸 때는 **Conv의 Bias를 끕니다.**

```python
# 추천하는 구조
layer = nn.Sequential(
    # bias=False 중요! (BN의 Beta가 Bias 역할을 대신 해주므로 중복 제거)
    nn.Conv2d(64, 128, kernel_size=3, bias=False),
    
    nn.BatchNorm2d(128),  # 여기서 중심 이동(Beta)을 담당함
    
    nn.ReLU(inplace=True)
)
```

In [85]:
layer =  nn.BatchNorm1d(3)
print(layer.weight) # 표준편차 역할
print(layer.bias) # 평균 역할
print("="*20)

x = torch.randn(5,3)
print(x)
print(layer(x))
print(layer(x).mean(dim=0))
print(layer(x).std(dim=0, unbiased=False)) # torch.std는 N-1로 나눔

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([[-0.4851, -1.0412, -0.2543],
        [ 0.2777,  0.4320,  0.0753],
        [-0.1760,  0.1739,  0.9581],
        [ 0.4550,  0.7288,  0.5125],
        [ 0.8756,  1.0091, -1.2068]])
tensor([[-1.4141, -1.8368, -0.3688],
        [ 0.1850,  0.2420,  0.0793],
        [-0.7661, -0.1222,  1.2793],
        [ 0.5568,  0.6608,  0.6736],
        [ 1.4384,  1.0563, -1.6635]], grad_fn=<NativeBatchNormBackward0>)
tensor([1.1921e-08, 1.1921e-08, 0.0000e+00], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


# ⚔️ Batch Norm(BN) vs Layer Norm(LN) 완벽 비교

두 코드는 **"정규화를 하는 방향(축)"**이 서로 정반대입니다.
`dim` 파라미터가 왜 다른지 직관적으로 정리했습니다.

---

## 1. 🏫 학교 성적표 비유 (직관)

데이터 `(5, 3)`을 **학생 5명의 3과목(국어, 수학, 영어) 성적표**라고 가정합시다.

### ① BatchNorm (`dim=0`): "전교 석차 따지기"
* **관점:** "철수가 수학을 80점 받았네. **다른 애들(Batch)에 비해** 잘 본 건가?"
* **방향:** **세로 방향 ($\downarrow$)** (과목별로 학생들을 줄 세움)
* **동작:** `dim=0` (Batch 축)을 압축하여 없앰.
* **결과:** 과목 수만큼의 평균이 나옴 (국어 평균, 수학 평균, 영어 평균).

### ② LayerNorm (`dim=1`): "내 적성 찾기 (자아 성찰)"
* **관점:** "철수가 수학 80점이네. **철수의 다른 과목(Feature) 점수에 비해** 잘 본 건가?"
* **방향:** **가로 방향 ($\rightarrow$)** (학생 혼자서 자기 과목끼리 비교)
* **동작:** `dim=1` (Feature 축)을 압축하여 없앰.
* **결과:** 학생 수만큼의 평균이 나옴 (철수 평균, 영희 평균...).

---

## 2. 💻 코드 분석: 왜 dim이 다를까?

### Case 1: BatchNorm (dim=0)
```python
# BN은 같은 과목(Channel)끼리 묶어서 계산합니다.
print(layer(x).mean(dim=0))
```
* **의미:** "학생 5명의 점수를 퉁쳐서 과목별 평균을 내라."
* **결과 Shape:** `[3]` (과목이 3개니까)

### Case 2: LayerNorm (dim=1)
```python
# LN은 한 학생(Sample) 안에서 모든 과목을 묶어서 계산합니다.
print(layer(x).mean(dim=1))
```
* **의미:** "과목 3개의 점수를 퉁쳐서 학생별 평균을 내라."
* **결과 Shape:** `[5]` (학생이 5명이니까)

---

## 3. 🔍 한 눈에 보는 요약표

| 구분 | **Batch Norm (BN)** | **Layer Norm (LN)** |
| :--- | :--- | :--- |
| **비유** | **절대 평가 (남과 비교)** | **자아 성찰 (나와 비교)** |
| **방향** | **세로 ($\downarrow$)** | **가로 ($\rightarrow$)** |
| **사라지는 차원** | `dim=0` (Batch) | `dim=1` (Feature) |
| **주 사용처** | 이미지 (CNN) | **자연어 (Transformer, LLM)** |

> **💡 왜 LLM은 LayerNorm을 쓸까?**
> 문장마다 길이가 다르고 배치 사이즈가 작아도, LN은 **"나 혼자(문장 1개)"** 평균을 내기 때문에 통계가 안정적입니다. 그래서 Transformer 계열은 무조건 LN을 씁니다.

In [86]:
layer = nn.LayerNorm(3)
print(layer.weight) # 표준편차 역할
print(layer.bias) # 평균 역할

x = torch.randn(5,3)
print(x)
print(layer(x))
print(layer(x).mean(dim=1))
print(layer(x).std(dim=1, unbiased=False))

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([[-0.1567,  0.2130,  0.2068],
        [ 1.1690, -2.0727, -1.4830],
        [-0.6112, -1.1054,  0.6505],
        [ 2.8051, -0.4510, -1.0562],
        [ 1.2804,  1.2033,  0.2162]])
tensor([[-1.4138,  0.7248,  0.6891],
        [ 1.3934, -0.9058, -0.4876],
        [-0.3460, -1.0145,  1.3605],
        [ 1.3991, -0.5211, -0.8780],
        [ 0.7852,  0.6260, -1.4112]], grad_fn=<NativeLayerNormBackward0>)
tensor([ 3.9736e-08,  1.9868e-08,  0.0000e+00,  0.0000e+00, -1.5895e-07],
       grad_fn=<MeanBackward1>)
tensor([0.9998, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


In [87]:
layer = nn.BatchNorm2d(3)
print(layer.weight)
print(layer.bias)

x = torch.randn(5,3,32,32)
print(layer(x).mean(dim=(0,2,3)))
print(layer(x).std(dim=(0,2,3),unbiased=False))

Parameter containing:
tensor([1., 1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
tensor([-2.2352e-09, -7.4506e-10,  2.2352e-09], grad_fn=<MeanBackward1>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<StdBackward0>)


# 📉 Dropout A to Z:

> **핵심 요약:** > "학습(Train) 때는 **강제로 연차(Dropout)**를 보내서 자생력을 기르고, 실전(Test) 때는 **전원 출근**시켜서 풀 파워를 낸다."

---

## 1. 🏢 직관: "회사가 잘 돌아가는 비결" (The Intuition)

### 💀 문제 상황: Co-adaptation (상호 의존)
* **상황:** 회사에 일을 엄청 잘하는 '김 대리' 한 명이 있습니다.
* **부작용:** 나머지 직원들은 김 대리만 믿고 묻어가려고 합니다. (Free-riding)
* **결과:** 김 대리가 아프면 회사가 마비됩니다. 딥러닝 모델로 치면 특정 노드에 **과적합(Overfitting)**된 상태입니다.

### 🛡️ 해결책: 강제 연차 (Dropout)
* **Training (연습):** 사장님이 매일 아침 동전을 던져서 **직원의 절반($p$)을 랜덤하게 집에 보냅니다.**
    * 남은 사람들끼리 어떻게든 일을 처리해야 하므로, **'김 대리' 없이도 돌아가는 법**을 배웁니다.
    * 직원들이 각자 **1인분 이상의 능력(개성 있는 특징)**을 갖게 됩니다.
* **Test (실전):** **"대통령이 방문했습니다!"** 👔
    * 중요한 날이니 **전원 출근**시킵니다.
    * 각자 능력이 올라간 상태에서 다 같이 일하니까 퍼포먼스가 최상이 됩니다. (앙상블 효과)

---

## 2. 🎚️ 메커니즘 & 수식: "성량 조절의 비밀"

연습 때는 50명만 노래 부르다가, 실전에서 100명이 다 부르면 **소리(출력값)가 2배로 커지는 문제**가 발생합니다. 이를 해결하는 방법은 두 가지입니다.

### 1) Original Paper 방식 (직관적)
* **Train:** 그냥 50명만 부름.
* **Test:** 인원이 2배가 됐으니, **"각자 목소리를 절반($\times p$)으로 줄여!"**라고 명령.
* *단점:* 테스트할 때마다 계산해야 해서 번거로움.

### 2) PyTorch 방식 (Inverted Dropout) ⭐ **[실제 사용]**
* **Train:** "너네 인원 적으니까, 연습 때 미리 **목소리를 2배($\times \frac{1}{p}$)로 키워서 불러!**" (Scaling Up)
* **Test:** **아무것도 건드리지 않음.** (그냥 전원 출근해서 평소대로 부름)
* *장점:* 실전(Inference) 속도가 빠르고 코드가 깔끔함.

---

## 3. 🖼️ 시각화: "노드의 개성" (Autoencoder 실험)

혁펜하임님이 MNIST(손글씨) 오토인코더의 은닉층(Hidden Layer)을 시각화했을 때의 차이입니다.

| 구분 | **Dropout 미적용 (Without)** | **Dropout 적용 (With)** |
| :--- | :--- | :--- |
| **이미지** | 자글자글한 노이즈 (TV 화면 지지직) | 뚜렷한 특징 (선, 곡선, 획) |
| **상태** | **눈치 게임 중 (Co-adaptation)**<br>"옆 사람이 해주겠지" 하고 대충 학습함. | **전문가 등극**<br>"내가 없으면 안 돼!"라는 마인드로<br>각자 확실한 특징(Feature)을 담당함. |

---

## 4. 💻 PyTorch 실전 코드 & 주의사항

### 핵심 코드
```python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    # p=0.5: 50% 확률로 노드를 꺼버림 (0으로 만듦)
    # PyTorch는 이때 살아남은 값을 2배로 튀기기(Scaling)까지 자동으로 해줌!
    nn.Dropout(p=0.5),
    nn.Linear(256, 10)
)
```

### 🚨 개발자의 악몽 (가장 많이 하는 실수)

> **Q. 똑같은 사진을 넣었는데 결과가 계속 바뀌어요! 왜 이러죠?**
> **A. `model.eval()`을 안 썼기 때문입니다.**

* **`model.train()`:** Dropout 켜짐 (랜덤 연차 + 목소리 뻥튀기 ON)
* **`model.eval()`:** Dropout 꺼짐 (전원 출근 + 목소리 뻥튀기 OFF)

👉 **추론(Inference)이나 테스트를 할 때는 반드시 `model.eval()`을 선언해야 합니다.**

---

## 5. 📝 3줄 요약

1.  **목적:** 특정 뉴런 편애(과적합)를 막고, 모든 뉴런을 **정예 요원(Feature Extractor)**으로 만들기 위함.
2.  **동작:** PyTorch는 **학습 때 값을 키워놓고(Inverted)**, 테스트 때는 아무 짓도 안 한다.
3.  **주의:** 실전에서 **`model.eval()`** 빼먹으면 사장님(사용자)한테 혼난다.

In [88]:
# Dropout은 논문과 구현이 다르다
x = torch.randn(3,7)
drop = nn.Dropout(p=0.7) # 구현에서 p는 죽일 확률

print(x)
print(drop(x)) # 구현은 반대로 훈련 때 1/살릴확률을 곱하고 테스트 때는 그대로

drop.eval()
print(drop(x))

"""
nn.Linear(10,100)
nn.BatcnhNorm1d(100)
nn.ReLU()
nn.Dropout(p=0.5) 이런 식으로 구성!
"""

tensor([[-0.9236, -1.0324, -0.9618,  0.3548,  2.1706, -0.5591,  0.5886],
        [ 0.0267,  1.2505,  1.3624, -1.2378, -1.2304, -0.6586,  0.2976],
        [ 0.7866, -0.7370,  1.7140,  0.9838,  0.8411, -0.2677, -0.5172]])
tensor([[-0.0000, -0.0000, -0.0000,  1.1828,  0.0000, -0.0000,  1.9622],
        [ 0.0000,  4.1683,  4.5415, -0.0000, -0.0000, -2.1955,  0.9919],
        [ 0.0000, -2.4566,  0.0000,  0.0000,  0.0000, -0.8922, -0.0000]])
tensor([[-0.9236, -1.0324, -0.9618,  0.3548,  2.1706, -0.5591,  0.5886],
        [ 0.0267,  1.2505,  1.3624, -1.2378, -1.2304, -0.6586,  0.2976],
        [ 0.7866, -0.7370,  1.7140,  0.9838,  0.8411, -0.2677, -0.5172]])


'\nnn.Linear(10,100)\nnn.BatcnhNorm1d(100)\nnn.ReLU()\nnn.Dropout(p=0.5) 이런 식으로 구성!\n'

In [89]:
class sample_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.drop_layer = nn.Sequential(
            nn.Linear(5,7),
            nn.ReLU(),
            nn.Dropout(p=0.9)
        )

    def forward(self,x):
        x = self.drop_layer(x)
        return x

model = sample_model()
model.train() # train mode로 전환
x = torch.randn(2,3,5)
print(model(x))

model.eval() # test mode
print(model(x)) # 일부 0인 이유: 음수인 애들이 ReLU 때문에 사라짐.

tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 4.4049, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.3472, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.2535, 0.0000, 0.0000]]],
       grad_fn=<MulBackward0>)
tensor([[[0.0000, 0.4798, 0.4121, 0.0000, 0.0000, 0.2265, 0.0000],
         [0.0000, 0.0753, 0.3716, 0.0000, 0.0000, 0.0000, 0.9997],
         [0.8194, 0.0000, 0.8512, 0.0000, 0.0000, 0.3130, 0.0000]],

        [[0.0000, 0.4405, 0.0000, 0.2788, 0.1285, 0.6235, 0.7894],
         [0.0000, 1.0445, 1.8149, 0.0000, 0.0347, 0.0812, 0.0000],
         [1.0014, 0.0000, 0.8237, 0.0000, 0.0253, 0.4565, 0.0000]]],
       grad_fn=<ReluBackward0>)


In [90]:
layer= nn.Conv2d(in_channels=1,out_channels=2,kernel_size=3) # stride=1, padding=0 이 디폴트
layer(torch.randn(32,1,5,5)).shape
# nn.Linear(3,5) # 채채 # 근데 얘는 채 또는 개채를 원함.  개X3 을 개X5로.
# nn.Conv2d(3,5) # 채채 # 근데 얘는 채행열 또는 개채행열을 원함. 개X3X행X열 을 개X5X행X열 로.

torch.Size([32, 2, 3, 3])

In [91]:
x = torch.randn(32,1,28,28)
conv1 = nn.Conv2d(1,8,6, stride=2)

# 예측해보기
print(conv1(x).shape)

conv2 = nn.Conv2d(8,16,3,padding=1)
# 예측해보기
print(conv2(conv1(x)).shape)

Maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
# 예측해보기
print(Maxpool(conv2(conv1(x))).shape)


torch.Size([32, 8, 12, 12])
torch.Size([32, 16, 12, 12])
torch.Size([32, 16, 6, 6])


In [92]:
maxpool = nn.MaxPool2d(2) # 2로만 줘도 자동 kernel_size=stride=2

x = torch.randn(1,6,6)
print(x)
# 예측해보기
print(maxpool(x))
# 예측해보기
print(maxpool(torch.randn(32,3,6,6)).shape)

tensor([[[-1.8288e+00, -8.7818e-01,  3.2074e-01,  1.7321e+00,  9.9866e-01,
           4.2583e-02],
         [ 1.1333e+00, -7.2772e-01, -1.0111e+00,  5.8496e-01, -2.2057e-01,
          -6.8198e-01],
         [-1.5464e+00,  1.5041e+00, -7.4079e-02, -9.8523e-01,  8.4480e-02,
          -2.1456e-03],
         [-1.2552e+00,  2.4836e+00, -5.2520e-01,  3.5551e-01, -1.1056e-02,
           1.0349e-02],
         [ 2.2337e+00,  4.5876e-01,  5.2749e-01, -2.8567e-01,  1.1901e+00,
          -9.2431e-01],
         [ 1.2854e+00,  9.8828e-01,  1.4502e+00, -3.6451e-01,  8.8503e-02,
          -7.0080e-01]]])
tensor([[[1.1333, 1.7321, 0.9987],
         [2.4836, 0.3555, 0.0845],
         [2.2337, 1.4502, 1.1901]]])
torch.Size([32, 3, 3, 3])


In [93]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.seq = nn.Sequential(
            nn.Conv2d(1,8,6, stride=2),
            nn.Conv2d(8,16,3, padding=1),
            nn.MaxPool2d(2)
            )

        # self.conv1 = nn.Conv2d(1,8,6, stride=2)
        # self.conv2 = nn.Conv2d(8,16,3, padding=1)
        # self.Maxpool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16*6*6,10)

    def forward(self, x):
        x= self.seq(x)
        # x = self.conv1(x)
        # x = self.conv2(x)
        # x = self.Maxpool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

x = torch.randn(32,1,28,28)
model = CNN()
print(model(x).shape)

torch.Size([32, 10])


# .parameters() & .modules() & .children()

# 1. model.parameters()

`model.parameters()`는 크게 **최적화, 전이학습, 디버깅** 이 3가지 상황에서 등장합니다.

---

## 1. 작업 반장(Optimizer)에게 "작업 지시서" 줄 때 (가장 기본) ⭐
Optimizer(Adam, SGD 등)를 만들 때, **"어떤 파라미터를 업데이트할지"** 명단을 넘겨줘야 합니다. 이때 가장 많이 씁니다.

```python
import torch.optim as optim

# "야, 이 모델에 있는 '모든' 파라미터를 다 학습(업데이트) 시켜!"
optimizer = optim.Adam(model.parameters(), lr=0.001)
```

---

## 2. 전이학습(Transfer Learning): "얼리고 녹이기" ❄️🔥
남이 만든 모델을 가져와서 내 데이터에 맞게 튜닝할 때, **전체를 얼리거나(Freeze) 일부만 녹일 때** 사용합니다.

### ① 전체 얼리기 (Freeze All)
```python
# 반복문으로 하나씩 꺼내서 "학습하지 마!(False)"라고 딱지 붙이기
for p in model.parameters():
    p.requires_grad = False
```

### ② 학습할 녀석들만 골라내기 (Filter)
전이학습 때는 **"학습해야 할(녹아있는) 애들"**만 골라서 Optimizer에 넣어줘야 효율적입니다.

```python
# 방법 1: "학습 가능한(True) 애들만 모아서 리스트로 만들어줘" (정석)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params, lr=0.1)

# 방법 2: "뒤에서 4개만 가져와!" (강의 꿀팁 - 구조를 알 때)
# (마지막 층 Weight, Bias + 그 앞 층 Weight, Bias = 총 4개)
optimizer = optim.Adam(list(model.parameters())[-4:], lr=0.1)
```

---

## 3. 내부 들여다보기 (Debugging & Counting) 🧐
모델이 학습이 잘 안 되거나, 파라미터 개수가 몇 개인지 궁금할 때 직접 까볼 수 있습니다.

```python
# 그냥 출력하면 generator 객체라 안 보임 -> list로 변환 필수!
param_list = list(model.parameters())

# 첫 번째 레이어의 Weight 값 확인
print(param_list[0])

# 전체 파라미터 개수 세기 (모델 크기 측정)
num_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 개수: {num_params}")
```

---

### 💡 3줄 요약
1.  **Optimizer** 만들 때 "얘네 학습시켜!" 하고 던져주는 용도.
2.  **전이학습** 할 때 `requires_grad`를 조작해서 얼리고 녹이는 용도.
3.  **디버깅** 할 때 가중치(Weight) 값이 정상인지 까보는 용도.

In [94]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1= nn.Sequential(nn.Linear(2,3),
                                nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(3,4),
                                 nn.ReLU())
        self.fc_out = nn.Sequential(nn.Linear(4,1),
                                    nn.Sigmoid())

    def forward(self,x):
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc_out(x)
        return x

model = MLP()
print(model(torch.randn(2,2)).shape)
print(model)

torch.Size([2, 1])
MLP(
  (fc1): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=True)
    (1): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=3, out_features=4, bias=True)
    (1): ReLU()
  )
  (fc_out): Sequential(
    (0): Linear(in_features=4, out_features=1, bias=True)
    (1): Sigmoid()
  )
)


In [95]:
model.parameters()

<generator object Module.parameters at 0x7e89e437b3e0>

In [96]:
list(model.parameters())
# [layer0 weight 값, layer0 bias 값, layer1 weight 값, layer1 bias 값, ...]

[Parameter containing:
 tensor([[-0.5965, -0.0511],
         [ 0.2305, -0.4897],
         [ 0.0800, -0.1281]], requires_grad=True),
 Parameter containing:
 tensor([-0.5202, -0.4077,  0.4193], requires_grad=True),
 Parameter containing:
 tensor([[ 0.2449,  0.4487, -0.4499],
         [ 0.1931, -0.4732, -0.3400],
         [-0.0812,  0.4658,  0.4536],
         [ 0.5140,  0.3341, -0.3334]], requires_grad=True),
 Parameter containing:
 tensor([ 0.4129, -0.4232, -0.0596,  0.3433], requires_grad=True),
 Parameter containing:
 tensor([[-0.2178,  0.0944,  0.2793, -0.3342]], requires_grad=True),
 Parameter containing:
 tensor([0.3661], requires_grad=True)]

In [97]:
# for transfer learning (마지막 layer만)
model = MLP()
print([p for p in model.parameters() if p.requires_grad])

# 일단 전체 freeze 시키기
for p in model.parameters():
    p.requires_grad = False

# 마지막 layer를 이진분류에서 10개 분류로 바꾸기
model.fc_out = nn.Linear(4,10)

# 마지막 layer freeze는 이미 풀려있음. 전이학습 시키기 (PyTorch의 레이어는 생성될 때 기본적으로 requires_grad=True)
params = [p for p in model.parameters() if p.requires_grad]
print(params)

from torch import optim
optimizer = optim.Adam(params, lr=0.1)

# 이후 실제 학습이 되려면 for 문을 돌면서 inference loss backpropagation update 과정을 거쳐야 함.

[Parameter containing:
tensor([[-0.1958, -0.3267],
        [ 0.4077,  0.4768],
        [ 0.0695,  0.4988]], requires_grad=True), Parameter containing:
tensor([ 0.5649, -0.3636, -0.3158], requires_grad=True), Parameter containing:
tensor([[ 0.0670,  0.4165, -0.1122],
        [ 0.2497,  0.2109, -0.2871],
        [-0.1993, -0.0296, -0.0414],
        [-0.4885, -0.5031,  0.5046]], requires_grad=True), Parameter containing:
tensor([ 0.3789, -0.4862, -0.3822, -0.4001], requires_grad=True), Parameter containing:
tensor([[-0.3764, -0.3451, -0.0522, -0.4840]], requires_grad=True), Parameter containing:
tensor([-0.4393], requires_grad=True)]
[Parameter containing:
tensor([[ 0.3290,  0.1248, -0.0999, -0.2064],
        [-0.4614,  0.1340,  0.1345, -0.1290],
        [-0.4026, -0.4367, -0.4339,  0.3026],
        [-0.4724, -0.0036,  0.0320, -0.2142],
        [ 0.2056, -0.0974,  0.2778,  0.4634],
        [-0.0510,  0.1165,  0.3795,  0.3389],
        [-0.1337, -0.4045, -0.3451, -0.1835],
        [ 0.0715

# 🔄 전이학습(Transfer Learning) 비교: 범위의 차이

두 코드는 **"어디까지 얼리고(Freeze), 어디부터 학습(Update)할 것인가"**의 결정적인 차이가 있습니다.

---

## 1. 코드 분석: `[:-4]`의 정체

### ① 첫 번째 코드 (Feature Extraction)
```python
# 전체 파라미터를 다 얼리고, 마지막 한 놈만 교체
for p in model.parameters():
    p.requires_grad = False
```
* **전략:** "건물 뼈대(1~9층)는 절대 건드리지 마! **옥상(10층)만** 새로 지어."
* **학습 범위:** **마지막 1개 Layer** (새로 만든 `fc_out`)

### ② 두 번째 코드 (Fine-tuning)
```python
# 뒤에서 4개 파라미터(Weight, Bias 2쌍)를 제외하고 얼림
for p in list(model.parameters())[:-4]:
    p.requires_grad = False
```
* **수식:** `Linear` 레이어 1개 = 파라미터 2개 (`Weight`, `Bias`)
    * `[:-4]` = 뒤에서 4개 제외 = **뒤에서 2개 Layer(마지막 층 + 바로 앞 층)를 남김.**
* **전략:** "옥상(10층) 새로 짓고, 그 바로 밑에 **9층 VIP룸(직전 은닉층)까지 리모델링** 해."
* **학습 범위:** **마지막 2개 Layer** (`fc_out` + `Hidden Layer`)

---

## 2. 🏗️ 비유: 단순 교체 vs 심화 공사

| 구분 | 첫 번째 코드 | 두 번째 코드 |
| :--- | :--- | :--- |
| **방식** | **Feature Extraction** | **Fine-tuning** (미세 조정) |
| **공사 범위** | **오직 막내(마지막 층)만** | **막내 + 사수(바로 윗 선임)** |
| **기존 지식** | "선배님들 지식은 완벽하니 건드리지 마." | "선배님도 이번 프로젝트에 맞춰서 **재교육** 좀 받으시죠." |
| **속도** | 빠름 (학습 파라미터 적음) | 상대적으로 느림 (학습 파라미터 많음) |

---

## 3. 🧠 왜 두 번째 방식(Fine-tuning)을 쓸까?

딥러닝 모델은 뒤쪽 층(Layer)으로 갈수록 **구체적이고 고차원적인 특징**을 봅니다.

* **앞단 (Freeze):** 선, 곡선, 색깔 등 기초 정보 (어떤 이미지든 비슷함 → 고정해도 됨)
* **뒷단 (Unfreeze):** 눈, 코, 입, 털의 질감 등 **데이터 특화 정보.**
* **결론:** 내 데이터셋에 더 정교하게 맞추고 싶다면, **뒷단 몇 개 층(`[:-4]` 등)을 열어서 같이 학습**시키는 것이 성능이 더 좋다.

In [98]:
# for transfer learning 2 (마지막 층 교체, 그 전 층과 함께 학습시키자.)
# 내 풀이
model = MLP()
print(list(model.parameters()))
print('==='*20)
print(list(model.parameters())[-4:])
print('==='*20)

# 일단 전체 layer freeze
for p in list(model.parameters()):
    p.requires_grad = False

# 마지막 층 교체
model.fc_out = nn.Linear(4,10)

# 마지막 layer 전 layer freeze 풀기
for p in list(model.parameters())[-4:]:
    p.requires_grad = True

from torch import optim
# optimizer에 학습할 파라미터 넘기기
optimizer = optim.Adam(list(model.parameters())[-4:], lr=0.1)


[Parameter containing:
tensor([[-0.3483,  0.4331],
        [ 0.5894,  0.4801],
        [ 0.3586,  0.0451]], requires_grad=True), Parameter containing:
tensor([-0.6054, -0.3648, -0.4877], requires_grad=True), Parameter containing:
tensor([[-0.2018, -0.3925,  0.0704],
        [-0.1596,  0.0836, -0.4462],
        [ 0.1910, -0.1033, -0.2038],
        [-0.0948,  0.1687, -0.1949]], requires_grad=True), Parameter containing:
tensor([-0.4987,  0.2378, -0.5322,  0.4483], requires_grad=True), Parameter containing:
tensor([[ 0.3963,  0.4491, -0.0672, -0.0681]], requires_grad=True), Parameter containing:
tensor([-0.1709], requires_grad=True)]
[Parameter containing:
tensor([[-0.2018, -0.3925,  0.0704],
        [-0.1596,  0.0836, -0.4462],
        [ 0.1910, -0.1033, -0.2038],
        [-0.0948,  0.1687, -0.1949]], requires_grad=True), Parameter containing:
tensor([-0.4987,  0.2378, -0.5322,  0.4483], requires_grad=True), Parameter containing:
tensor([[ 0.3963,  0.4491, -0.0672, -0.0681]], requires_gr

# 2. model.modules()

# 📦 `model.modules()` 핵심 완전 정복

> **핵심 요약:** `model.modules()`는 모델 안에 있는 **모든 부품(Layer, Block, Container)**을 **재귀적으로(Recursive)** 하나하나 다 끄집어내서 보여주는 만능 도구입니다.

---

## 1. 🔍 `model.modules()`의 정체: "양파 같은 녀석"

`model.modules()`는 단순히 겉에 있는 레이어만 보여주는 게 아니라, **속에 있는 것까지 다 까서** 보여줍니다.

### 🧅 탐색 순서 (깊이 우선 탐색 느낌)
1.  **전체 모델 (껍데기)**
2.  **첫 번째 덩어리 (예: Sequential1)**
    * 그 안의 첫 번째 부품 (예: Linear)
    * 그 안의 두 번째 부품 (예: ReLU)
3.  **두 번째 덩어리 (예: Sequential2)**
    * 그 안의 첫 번째 부품...

> **주의:** 그래서 `list(model.modules())`를 찍어보면 리스트 길이가 생각보다 엄청 깁니다. (전체도 나오고, 부분도 나오고, 낱개도 나오니까요.)

---

## 2. 🛠️ 핵심 용도: "가중치 초기화 (Weight Initialization)" ⭐⭐⭐

이게 제일 중요합니다.
**"야, 모델 돌면서 `Linear` 층만 찾아서, 가중치를 '카이밍(Kaiming)' 방식으로 초기화해!"** 라고 시킬 때 씁니다.

### 💻 필수 코드 패턴 (암기 추천)

```python
import torch.nn as nn

# 모델의 모든 부품을 하나씩 꺼내면서 반복문 돎
for m in model.modules():
    
    # Q. 너 정체가 뭐냐? Linear 층이냐? (isinstance 활용)
    if isinstance(m, nn.Linear):
        # A. 맞습니다! -> 그럼 초기화 진행시켜.
        
        # 1. Weight는 'kaiming_normal'로 초기화
        nn.init.kaiming_normal_(m.weight)
        
        # 2. Bias는 그냥 '0'으로 초기화 (상수로 밀기)
        nn.init.constant_(m.bias, 0)
        
    # Q. 아니면 너 혹시 BatchNorm 층이냐?
    elif isinstance(m, nn.BatchNorm1d):
        # BN은 Weight를 1로, Bias를 0으로 하는 게 국룰
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
```

> **꿀팁 (`_`의 비밀):** `kaiming_normal_` 처럼 함수 뒤에 **언더바(`_`)**가 붙으면 **"In-place Operation"**입니다. 즉, 새로운 변수를 만들지 않고 **그 자리에서 바로 값을 바꿔치기**한다는 뜻입니다.

---

## 3. 🆚 `model.children()`과의 차이점

이거 은근히 면접 질문으로도 나옵니다.

| 구분 | **`model.modules()`** | **`model.children()`** |
| :--- | :--- | :--- |
| **탐색 범위** | **속의 속까지 다 깜 (재귀적)** | **겉에 있는 자

In [99]:
model.modules()

<generator object Module.modules at 0x7e8a252b0a00>

In [100]:
list(model.modules())

[MLP(
   (fc1): Sequential(
     (0): Linear(in_features=2, out_features=3, bias=True)
     (1): ReLU()
   )
   (fc2): Sequential(
     (0): Linear(in_features=3, out_features=4, bias=True)
     (1): ReLU()
   )
   (fc_out): Linear(in_features=4, out_features=10, bias=True)
 ),
 Sequential(
   (0): Linear(in_features=2, out_features=3, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=2, out_features=3, bias=True),
 ReLU(),
 Sequential(
   (0): Linear(in_features=3, out_features=4, bias=True)
   (1): ReLU()
 ),
 Linear(in_features=3, out_features=4, bias=True),
 ReLU(),
 Linear(in_features=4, out_features=10, bias=True)]

In [101]:
# model.modules 사용해서 nn.Linear 사용하는 layer 프린트하기
for m in model.modules():
    if isinstance(m,nn.Linear):
        print(m)

'''
Q: 재귀적으로 호출된다면, 부모를 부를 때 그 안의 자식도 보일 거고, 자식 차례 때 또 보일 테니 두 번 나와야 하는 거 아니야?

⚡️ 3줄 요약
1. model.modules()는 부모(Sequential)도 방문하고 자식(Linear)도 방문하는 게 맞다.

2. 하지만 **부모(Sequential)**는 Linear 클래스가 아니라서 if 문(isinstance(m,nn.Linear))에서 걸러진다.

3. 그래서 진짜 Linear 객체 순서가 되었을 때만 딱 한 번 출력되는 것이다.
'''

Linear(in_features=2, out_features=3, bias=True)
Linear(in_features=3, out_features=4, bias=True)
Linear(in_features=4, out_features=10, bias=True)


'\nQ: 재귀적으로 호출된다면, 부모를 부를 때 그 안의 자식도 보일 거고, 자식 차례 때 또 보일 테니 두 번 나와야 하는 거 아니야?\n\n⚡️ 3줄 요약\n1. model.modules()는 부모(Sequential)도 방문하고 자식(Linear)도 방문하는 게 맞다.\n\n2. 하지만 **부모(Sequential)**는 Linear 클래스가 아니라서 if 문(isinstance(m,nn.Linear))에서 걸러진다.\n\n3. 그래서 진짜 Linear 객체 순서가 되었을 때만 딱 한 번 출력되는 것이다.\n'

In [102]:
print([m for m in model.modules() if isinstance(m,nn.Linear)])

[Linear(in_features=2, out_features=3, bias=True), Linear(in_features=3, out_features=4, bias=True), Linear(in_features=4, out_features=10, bias=True)]


# 3. model.children()

# 👶 `model.children()` 핵심 정리

> **한 줄 요약:** 재귀적으로 들어가지 않고, **가장 바깥쪽(Top-level) 레이어**들만 리스트로 뽑아준다.

## 1. `modules()` vs `children()`
* **`modules()`:** 양파 까듯이 **속의 속(Linear, ReLU 등)**까지 다 보여줌. (가중치 초기화용)
* **`children()`:** **직속 덩어리(Block 단위)**만 보여줌. (모델 개조용)

## 2. 언제 쓰나? (Sub-network)
특정 덩어리만 떼어내서 **새로운 모델**을 만들거나, **특정 구간만 통과**시키고 싶을 때 사용.

```python
# 예: 기존 모델의 앞쪽 2개 블록만 가져와서 새 모델 만들기
layers = list(model.children())[:2]
new_model = nn.Sequential(*layers)
```

In [103]:
model = MLP()
print(model.children())
for c in list(model.children()):
    print(c)

<generator object Module.children at 0x7e8a252b1220>
Sequential(
  (0): Linear(in_features=2, out_features=3, bias=True)
  (1): ReLU()
)
Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
)
Sequential(
  (0): Linear(in_features=4, out_features=1, bias=True)
  (1): Sigmoid()
)


In [104]:
# model의 첫번째 sequential 덩어리에만 x넣기
x = torch.randn(2,2)
list(model.children())[0](x)

tensor([[0., 0., 0.],
        [0., 0., 0.]], grad_fn=<ReluBackward0>)

In [105]:
print(list(model.children())[:2]) # 리스트로 싸여있음
print(*list(model.children())[:2]) # *로 겉 리스트 unpacking

[Sequential(
  (0): Linear(in_features=2, out_features=3, bias=True)
  (1): ReLU()
), Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
)]
Sequential(
  (0): Linear(in_features=2, out_features=3, bias=True)
  (1): ReLU()
) Sequential(
  (0): Linear(in_features=3, out_features=4, bias=True)
  (1): ReLU()
)


In [106]:
# 앞에 두 덩어리만 sub_network에 넣기.
sub_network = nn.Sequential(*list(model. children())[:2])
print(sub_network)
sub_network(x)

Sequential(
  (0): Sequential(
    (0): Linear(in_features=2, out_features=3, bias=True)
    (1): ReLU()
  )
  (1): Sequential(
    (0): Linear(in_features=3, out_features=4, bias=True)
    (1): ReLU()
  )
)


tensor([[0.5305, 0.3000, 0.0000, 0.0000],
        [0.5305, 0.3000, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)

# Sequential vs ModuleList

# 🥊 `nn.Sequential` vs `nn.ModuleList` 핵심 정리

> **한 줄 요약:**
> **`nn.Sequential`**은 "직진만 가능한 **자동 변속기(Auto)**"이고,
> **`nn.ModuleList`**는 "내가 마음대로 기어를 넣는 **수동 변속기(Manual)**"입니다.

---

## 1. 두 클래스의 결정적 차이 (Difference)

| 특징 | **`nn.Sequential`** (편리함) | **`nn.ModuleList`** (자유로움) |
| :--- | :--- | :--- |
| **`forward` 함수** | **있음 (내장됨)** | **없음 (내가 짜야 함)** |
| **작동 방식** | 입력 1개를 받아서 **순서대로 자동 통과**시킴. | 그냥 **부품(Layer)을 보관**만 함. (아무 동작 안 함) |
| **연결 구조** | 무조건 **일자형 (Linear)** 연결. ($1 \to 2 \to 3$) | **내 맘대로.** (반복, 건너뛰기, 순서 바꾸기 가능) |
| **입출력 개수** | **Only 1개** (Input $\to$ Output) | **제한 없음** ($N$개 Input $\to$ $M$개 Output) |

---

## 2. 왜 `nn.ModuleList`를 써야 하는가? (사용 시점)

방금 작성하신 코드가 바로 **`ModuleList`가 필수적인 상황**입니다. `Sequential`로는 절대 구현할 수 없는 3가지 상황이 있습니다.

### ① 입력이 2개 이상일 때 (Multi-Input) ⭐ [작성하신 코드 상황]
`nn.Sequential`의 내장 `forward`는 `input`을 하나만 받도록 설계되어 있습니다.
```python
# Sequential: "야, x랑 y 두 개 던지면 나보고 어쩌라고?" (Error!)
model(x, y)

# ModuleList: "일단 보관해둘게. 꺼내서 어떻게 쓸지는 네가 for문 안에서 정해."
for block in model:
    x, y = block(x, y) # OK!
```

### ② 중간에 다른 길로 샐 때 (Skip Connection / Branching)
ResNet처럼 갔던 길을 되돌아와서 더하거나(Add), 중간 결과를 따로 저장해야 할 때.
* **Sequential:** 무조건 직진이라 중간 값을 못 빼옵니다.
* **ModuleList:** 인덱싱(`model[0]`)으로 중간 결과를 변수에 저장할 수 있습니다.

### ③ 같은 블록을 반복해서 쓸 때 (Iterative)
똑같은 레이어를 100번 통과시켜야 한다면?
* **Sequential:** `Sequential(layer, layer, layer...)` 100번 써야 함. (비효율)
* **ModuleList:** 그냥 `for i in range(100):` 돌리면 끝.

---

## 3. ⚠️ 주의: "그럼 그냥 Python List(`[]`) 쓰면 안 돼?"

**절대 안 됩니다!** (가장 중요한 포인트)

* **Python List:** `layers = [block(), block()]`
    * 파이토치가 이 리스트 안에 있는 게 신경망 부품인지 모릅니다.
    * **결과:** `model.parameters()`에 등록이 안 돼서 **학습(미분)이 안 됩니다.**
* **`nn.ModuleList`:**
    * "야 파이토치! 이 리스트 안에 있는 거 우리 식구(학습할 파라미터)니까 꼭 챙겨!"라고 **신고(Registration)**를 해줍니다.

---

## 🎓 결론

1.  **단순한 모델(LeNet, AlexNet 등):** 그냥 `nn.Sequential` 쓰세요. 그게 편합니다.
2.  **복잡한 모델(ResNet, Transformer, Multi-modal):** 입력이 많거나 구조가 복잡하면 **`nn.ModuleList`**에 담아두고 **`forward` 함수를 직접(for문)** 짜세요.

In [107]:
# Sequential과 ModuleList input 전달 방식의 차이
fc = nn.Linear(3,3)
layer_list = [fc for _ in range(5)]
layers1 = nn.Sequential(*layer_list)
layers2 = nn.ModuleList(layer_list) # ModuleLIst는 Sequential과 달리 리스트 언패킹을 안해줘도 된다.

print(layers1)
print(layers2)

x = torch.randn(1,3)
print(layers1(x))

# print(layers2(x)) # 대신 이렇게 주면 안되고
for layer in layers2:
    x = layer(x)
print(x) # 이렇게 하나하나 넣어줘야함.

Sequential(
  (0): Linear(in_features=3, out_features=3, bias=True)
  (1): Linear(in_features=3, out_features=3, bias=True)
  (2): Linear(in_features=3, out_features=3, bias=True)
  (3): Linear(in_features=3, out_features=3, bias=True)
  (4): Linear(in_features=3, out_features=3, bias=True)
)
ModuleList(
  (0-4): 5 x Linear(in_features=3, out_features=3, bias=True)
)
tensor([[-0.5163, -0.5442,  0.2604]], grad_fn=<AddmmBackward0>)
tensor([[-0.5163, -0.5442,  0.2604]], grad_fn=<AddmmBackward0>)


In [111]:
# 2. 굳이 nn.ModuleLIst를 써야하는 이유
class small_block(nn.Module):
    def __init__(self):
        super().__init__()
        self.block_x = nn.Linear(1,1)
        self.block_y = nn.Linear(1,1)
    def forward(self,x,y):
        x = self.block_x(x)
        y = self.block_y(y)
        return x,y

model = nn.Sequential(small_block(),
                      small_block(),
                      small_block())

# model(torch.randn(1),torch.randn(1)) # error!
# nn.Sequential이 가지고 있는 forward 함수는 무조건 인자를 '한 개'를 받기때문에 오류가 난다.

model = nn.ModuleList([small_block(),
                       small_block(),
                       small_block()
                    ])
x = torch.randn(1)
y = torch.randn(1)
for block in model:
    x, y = block(x,y)
print(x,y)

tensor([-0.0107], grad_fn=<ViewBackward0>) tensor([0.0812], grad_fn=<ViewBackward0>)


In [113]:
# 3. nn.ModuleList 대신 그냥 리스트를 쓰면 안되는 이유
class testNet(nn.Module):
    def __init__(self):
        super().__init__()

        # self.Module_List = [nn.Linear(3,3),nn.Linear(3,3)]
        self.Module_List = nn.ModuleList([nn.Linear(3,3), nn.Linear(3,3)])

    def forward(self,x):
        for layer in self.Module_List:
            x = layer(x)
        return x

model = testNet()
print(model(torch.randn(1,3))) # 통과까진 문제 없음

print(model) # 그냥 리스트로 하면 아무것도 등록 안되어있음.

optimizer = optim.Adam(model.parameters(),lr=0.1) # 등록 안되어있으면 parameter를 못 찾는다.

tensor([[-0.9167, -0.1752, -0.2817]], grad_fn=<AddmmBackward0>)
testNet(
  (Module_List): ModuleList(
    (0-1): 2 x Linear(in_features=3, out_features=3, bias=True)
  )
)
